<a href="https://colab.research.google.com/github/rubyratcha-19/GE338_lab5/blob/main/Lab5_6606614805.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### LAB5 PROJECT: AVATAR NA'VI HABITAT SUITABILITY MODEL (THAILAND)

6606614805 รัชชานนท์ สุรินทร์

In [1]:
# ---------------------------------------------------------
# 1. ตั้งค่าเริ่มต้นและเชื่อมต่อ Earth Engine
# ---------------------------------------------------------
!pip install geemap -q
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='inlaid-reactor-457503-u6') # Project ID ของคุณ

print("✅ เชื่อมต่อสำเร็จ!...")

# ---------------------------------------------------------
# 2. กำหนดพื้นที่ (ประเทศไทย) และดึงข้อมูลพื้นฐาน (2020-2023)
# ---------------------------------------------------------
Map = geemap.Map()
thailand = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))
Map.centerObject(thailand, 6)

start_date = '2020-01-01'
end_date = '2023-12-31'

# 1. ข้อมูลสภาพแวดล้อม (NDVI, ฝน, อุณหภูมิ, ความสูง)
elevation = ee.Image('USGS/SRTMGL1_003').clip(thailand)
elev_norm = elevation.unitScale(0, 2500).clamp(0, 1)

ndvi = ee.ImageCollection('MODIS/061/MOD13A1').filterDate(start_date, end_date).select('NDVI').mean().multiply(0.0001).clip(thailand)
ndvi_norm = ndvi.unitScale(0, 0.9).clamp(0, 1)

rainfall = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filterDate(start_date, end_date).sum().divide(4).clip(thailand)
rain_norm = rainfall.unitScale(1000, 4000).clamp(0, 1)

temp = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR").filterDate(start_date, end_date).select('temperature_2m').mean().subtract(273.15).clip(thailand)
temp_norm = temp.unitScale(20, 30).clamp(0, 1)

landcover = ee.ImageCollection("ESA/WorldCover/v200").first().clip(thailand)

# กำหนดให้พิกัดที่มีค่าเท่ากับ 10 (Class 10 = Trees / ป่าไม้) มีค่าเป็น 1 นอกนั้นเป็น 0
forest_mask = landcover.eq(10)



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.3 MB/s eta 0:00:00
✅ เชื่อมต่อสำเร็จ!...


In [ ]:

# 3. สร้างฟังก์ชัน: Carrying Capacity (พื้นที่สำหรับ 300 คน)

def apply_population_rule(suitability_image, min_score=0.6, min_pixels=300):
    """
    คัดกรองเฉพาะพื้นที่ที่คะแนนเกิน min_score
    และต้องมีขนาดพื้นที่ติดกันไม่น้อยกว่า min_pixels (ประมาณ 300 ตร.กม.)
    """
    habitable = suitability_image.gt(min_score)
    patch_size = habitable.connectedPixelCount(min_pixels, False)
    # คืนค่าแผนที่เดิม แต่ซ่อน (Mask) พื้นที่ที่เล็กเกินไป
    return suitability_image.updateMask(patch_size.gte(min_pixels))

# ---------------------------------------------------------
# 4. วิเคราะห์เผ่าป่า (Omaticaya: Forest Na'vi)
# ---------------------------------------------------------
# ใช้ ESA WorldCover คัดกรองเฉพาะ "พื้นที่ป่าจริงๆ" (Class 10)
landcover = ee.ImageCollection("ESA/WorldCover/v200").first().clip(thailand)
forest_mask = landcover.eq(10)

base_forest = ndvi_norm.multiply(0.4).add(rain_norm.multiply(0.3)).add(temp_norm.multiply(0.2)).add(elev_norm.multiply(0.1))
strict_forest = base_forest.updateMask(forest_mask) # ตัดเมือง/ทุ่งนาทิ้ง

# เอาเข้ากฎประชากร 300 คน
final_forest = apply_population_rule(strict_forest)

# ---------------------------------------------------------
# 5. วิเคราะห์เผ่าภูเขา (Ash/Fire: Mountain Na'vi)
# ---------------------------------------------------------

slope_norm_easy = slope.unitScale(0, 15).clamp(0, 1)
base_mountain = elev_norm.multiply(0.5).add(slope_norm_easy.multiply(0.5))
strict_mountain = base_mountain.updateMask(elevation.gt(150))

# และใช้พื้นที่ขั้นต่ำ 200 พิกเซล สำหรับเผ่าที่คนน้อยกว่า 300 คน
final_mountain = apply_population_rule(strict_mountain, min_score=0.5, min_pixels=200)
# ---------------------------------------------------------
# 6. วิเคราะห์เผ่าน้ำ (Metkayina: Ocean Na'vi)
# ---------------------------------------------------------
# หาชายฝั่งทะเล และรัศมีไม่เกิน 40 กม.
ocean_mask = elevation.lte(0)
dist_to_coast = ocean_mask.distance(ee.Kernel.euclidean(40000, 'meters'))
coastal_prox = ee.Image(1).subtract(dist_to_coast.divide(40000)).clamp(0, 1)

base_ocean = coastal_prox.multiply(0.8).add(ee.Image(1).subtract(elev_norm).multiply(0.2))
strict_ocean = base_ocean.updateMask(coastal_prox.gt(0)) # ตัดพื้นที่ตอนในประเทศทิ้ง

# เอาเข้ากฎประชากร 300 คน
final_ocean = apply_population_rule(strict_ocean, min_score=0.5, min_pixels=400)

# ---------------------------------------------------------
# 7. แสดงผลบนแผนที่ (Visualization)
# ---------------------------------------------------------
# พาเลตต์สี: แดง (เหมาะน้อย) -> ส้ม -> เหลือง -> เขียวอ่อน -> เขียวเข้ม (เหมาะสุดๆ)
vis_params = {'min': 0, 'max': 1, 'palette': ['red', 'orange', 'yellow', 'limegreen', 'darkgreen']}

# เพิ่มเลเยอร์เข้าแผนที่ (ซ่อนป่ากับภูเขาไว้ก่อน โชว์เผ่าน้ำเป็นหน้าแรก สามารถติ๊กเปิด/ปิดได้มุมขวาบน)
Map.addLayer(final_forest, vis_params, "Omaticaya_Forest", False)
Map.addLayer(final_mountain, vis_params, "AshTribe_Mountain", False)
Map.addLayer(final_ocean, vis_params, "Metkayina_Ocean", True)

# ตีกรอบประเทศไทย
empty_style = {'fillColor': '00000000', 'color': 'black', 'width': 1.5}
Map.addLayer(thailand.style(**empty_style), {}, 'Thailand Boundary')

Map

Map(center=[15.068739188750715, 101.01185086654338], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
# ---------------------------------------------------------
# 📤 ส่งออกภาพแผนที่ "ทั้ง 3 เผ่า" กลับไปเก็บใน GEE Assets
# ---------------------------------------------------------

print("⏳ กำลังส่งคำสั่ง Export ไปที่ Google Earth Engine...")

# 1. ส่งออกเผ่าป่า (Omaticaya)
export_forest = ee.batch.Export.image.toAsset(
    image=final_forest,
    description='Omaticaya_Forest_Map',
    assetId='projects/v6-dk-459909/assets/Omaticaya_Forest', # ชื่อไฟล์เผ่าป่า
    region=thailand.geometry(),
    scale=1000, # ความละเอียด 1 กม. ต่อ 1 พิกเซล (ปรับเป็น 100 ได้ถ้าอยากได้ภาพชัดๆ)
    maxPixels=1e10
)
export_forest.start()

# 2. ส่งออกเผ่าภูเขา (Ash Tribe)
export_mountain = ee.batch.Export.image.toAsset(
    image=final_mountain,
    description='AshTribe_Mountain_Map',
    assetId='projects/v6-dk-459909/assets/AshTribe_Mountain', # ชื่อไฟล์เผ่าภูเขา
    region=thailand.geometry(),
    scale=1000,
    maxPixels=1e10
)
export_mountain.start()

# 3. ส่งออกเผ่าน้ำ (Metkayina)
export_ocean = ee.batch.Export.image.toAsset(
    image=final_ocean,
    description='Export_Metkayina_Ocean_Map',
    assetId='projects/v6-dk-459909/assets/Metkayina_Ocean', # ชื่อไฟล์เผ่าน้ำ
    region=thailand.geometry(),
    scale=1000,
    maxPixels=1e10
)
export_ocean.start()

print("✅ สั่งรัน Export ทั้ง 3 เผ่าเรียบร้อยแล้ว!")

⏳ กำลังส่งคำสั่ง Export ไปที่ Google Earth Engine...
✅ สั่งรัน Export ทั้ง 3 เผ่าเรียบร้อยแล้ว!


In [ ]:
# ---------------------------------------------------------
# 📤 แปลงเป็นโพลิกอนและส่งออก Shapefile เข้า Google Drive (สำหรับ ArcGIS Pro)
# ---------------------------------------------------------

print("⏳ กำลังแปลงพิกเซลเป็นโพลิกอน และส่งเข้า Google Drive...")

# สร้างฟังก์ชันสำหรับแปลง Raster เป็น Vector และ Export
def export_to_shapefile(image, tribe_name):

    # 1. สร้างพื้นที่ Binary (เอาเฉพาะพื้นที่ที่ผ่านเกณฑ์/มีคะแนน)
    # พิกเซลที่ผ่านเกณฑ์จะเป็น 1 ส่วนที่ถูก Mask ทิ้งจะเป็น 0
    binary_image = image.gt(0)

    # 2. แปลงกลุ่มพิกเซล 1 ให้กลายเป็น Vector Polygon
    vector_data = binary_image.reduceToVectors(
        geometry=thailand.geometry(),
        crs='EPSG:4326',       # ระบบพิกัดมาตรฐานสากล (WGS84)
        scale=1000,            # ความละเอียด (1 กม. ต่อ 1 พิกเซล)
        geometryType='polygon',
        eightConnected=True,   # ลากเส้นเชื่อมโพลิกอนทั้ง 8 ทิศทาง
        maxPixels=1e10
    )

    # 3. สั่ง Export เป็น Shapefile ไปลง Google Drive
    task = ee.batch.Export.table.toDrive(
        collection=vector_data,
        description=f'Export_SHP_{tribe_name}',
        folder='Avatar_ArcPro',            # ชื่อโฟลเดอร์ที่จะไปโผล่ใน Google Drive
        fileNamePrefix=f'{tribe_name}_Map', # ชื่อไฟล์
        fileFormat='SHP'
    )
    task.start()
    print(f"📦 สั่งคิว Export เผ่า {tribe_name} เรียบร้อย!")

# เรียกใช้ฟังก์ชันส่งออกทั้ง 3 เผ่า
export_to_shapefile(final_forest, "Omaticaya_Forest")
export_to_shapefile(final_mountain, "Ash_Mountain")
export_to_shapefile(final_ocean, "Metkayina_Ocean")

print("✅ ส่งคำสั่งทั้งหมดเข้าเซิร์ฟเวอร์เรียบร้อย!")

⏳ กำลังแปลงพิกเซลเป็นโพลิกอน และส่งเข้า Google Drive...
📦 สั่งคิว Export เผ่า Omaticaya_Forest เรียบร้อย!
📦 สั่งคิว Export เผ่า Ash_Mountain เรียบร้อย!
📦 สั่งคิว Export เผ่า Metkayina_Ocean เรียบร้อย!
✅ ส่งคำสั่งทั้งหมดเข้าเซิร์ฟเวอร์เรียบร้อย!
